# Multi-LoRA SFT launcher (`pipeline.multi_lora_sft`)

Run all cells top to bottom. You only edit two cells:

1. **RUN** — pick a `preset` (or leave `"auto"`), training files, resume path.
2. **PARAMS** — every tunable hyperparameter. `None` keeps the preset / YAML default; any other value is passed as
   `--set key=value` and validated *before* anything expensive starts.

| Situation | `preset` | What it does |
| :--- | :--- | :--- |
| No GPU, just check the code path | `smoke_cpu` | tiny Gemma-4 fixture on CPU, 2 steps, builds a real `submission.zip` |
| E4B on GPUs, measure speed/memory first | `e4b_gpu_calibration` | 30-min run, both adapters, parity + smoke eval |
| E4B on GPUs, full session | `e4b_gpu_full` | bf16 LoRA, DDP over all GPUs |
| E4B on a TPU VM | `e4b_tpu` | torch_xla SPMD (FSDPv2) over all chips, static shapes, eval off |
| 31B on 4×L4 | `31b_gpu_full` | NF4 QLoRA, model split across GPUs |
| Continue a killed / previous run | any + `RUN["resume_from"]` | restores LoRA, optimizer, scheduler, RNG, data order |
| Don't know | `auto` | chooses from detected hardware (TPU → `e4b_tpu`, ≥4 big GPUs → `31b_gpu_full`, GPU → `e4b_gpu_full`, none → `smoke_cpu`) |

**When something goes wrong**

* *CUDA/TPU out of memory* — the orchestrator already retries with `train.oom_fallbacks` (shorter sequences / NF4).
  To avoid it up front: lower `train.max_seq_length`, keep `per_device_batch=1` and raise `grad_accum`.
* *Loss becomes NaN* — training aborts after `train.max_nonfinite_logs` bad logs and restarts from the last good checkpoint;
  lower `train.lr` or raise `schedule.warmup_steps`.
* *Session about to end* — phases are killed at their deadline with SIGTERM; training saves its adapter first.
  Re-run with `RUN["resume_from"]` pointing at the previous run directory to continue.
* *Model won't load* — the setup phase checks the checkpoint directory (config, shards, quantization type,
  train/served size match) and names the exact problem in `REPORT.md`.


In [ ]:
import os, time
os.environ["SESSION_T0"] = str(time.time())  # session clock starts here; every phase deadline counts from it

RUN = {
    "preset": "auto",          # auto | smoke_cpu | e4b_gpu_calibration | e4b_gpu_full | e4b_tpu | 31b_gpu_full
    "params": {},              # extra PARAMS overrides, e.g. {"train.lr": 2e-4} (handy with kaggle_push --set)
    "train_files": [],         # training JSONL paths; empty = auto-discover trajectory *.jsonl under /kaggle/input
    "eval_task_ids": [],       # fixed held-out task ids for the 3-task smoke eval
    "resume_from": None,       # previous run dir (or checkpoints/<adapter>/step_N) to continue training from
    "set": [],                 # raw "dotted.key=value" overrides, applied last
    "auto_adjust": True,       # apply the hardware / time / disk rules in the "Situational adjustments" cell
    "dry_run": False,          # validate + print the resolved config, then stop before training
    "git_sha": None,
}
RUN_OVERRIDES = {}  # @@RUN_OVERRIDES@@  (rewritten by scripts/kaggle_push.py)
RUN.update(RUN_OVERRIDES)
print(RUN)


## Hyperparameters

`None` = keep the preset / YAML value. Global `lora.*` / `train.*` values apply to every adapter unless the adapter has
its own override in the YAML (e.g. `tool_lora` uses r=16, 8k context) — use `ADAPTER_OVERRIDES` to change one adapter.
`submission.generation.*` is the **served agent's** decoding (temperature, thinking budget) written into the zip;
`submission.eval_budget.*` is the per-task scorer budget (`eval_config.yaml`).


In [ ]:
PARAMS = {
    # ---------------- Model ----------------
    "model.served_model_id": None,         # gemma-4-31b-it-qat-w4a16-ct | gemma-4-e4b-it-qat-w4a16-ct | gemma-4-e2b-... | gemma-4-12b-...
    "model.quantization": None,            # nf4 (QLoRA, CUDA only) | bf16 | fp32 (CPU smoke only)
    "model.attn_implementation": None,     # sdpa | eager | flash_attention_2 (CUDA only)
    "model.max_memory_gib": None,          # per-GPU cap used by model_split placement (31B)
    # ---------------- Hardware ----------------
    "train.accelerator": None,             # auto | cuda | tpu | cpu
    "train.parallelism": None,             # model_split (31B) | ddp (E4B on GPUs) | single | spmd (TPU, all chips)
    "train.num_processes": None,           # DDP process count (default = number of GPUs)
    # ---------------- LoRA ----------------
    "lora.r": None,                        # rank 1..128 (served max_lora_rank = highest rank in the zip)
    "lora.alpha": None,                    # usually 2 * r
    "lora.dropout": None,
    "lora.use_rslora": None,
    # ---------------- Optimisation ----------------
    "train.lr": None,                      # peak LR of the warmup-stable-decay schedule
    "train.max_epochs": None,
    "train.max_steps": None,               # hard step cap (None = epochs / time budget decide)
    "train.max_seq_length": None,          # longer rows are dropped; main OOM lever
    "train.per_device_batch": None,        # SPMD (TPU): this is the GLOBAL batch, multiple of the chip count
    "train.grad_accum": None,
    "target_effective_batch": None,        # helper: derive grad_accum from per_device_batch x data-parallel size
    "train.weight_decay": None,
    "train.max_grad_norm": None,
    "schedule.warmup_steps": None,
    "schedule.decay_frac": None,           # fraction of the time budget spent decaying the LR at the end
    "schedule.final_lr_ratio": None,
    # ---------------- Loss ----------------
    "loss.impl": None,                     # selective (GPU) | static (TPU / XLA)
    "loss.chunk_rows": None,               # rows per logits chunk; lower = less memory
    "loss.checkpoint_chunks": None,        # recompute chunk logits in backward (big memory saver)
    "loss.tool_call_weight": None,         # >1 up-weights tool-call tokens
    # ---------------- Checkpointing & robustness ----------------
    "train.checkpoint_every_min": None,
    "train.keep_checkpoints": None,
    "train.val_every_min": None,
    "train.max_restarts": None,            # crash restarts from the newest checkpoint
    "train.max_nonfinite_logs": None,      # NaN/Inf logs in a row before rolling back to the last checkpoint
    "train.min_free_disk_gib": None,       # skip a checkpoint instead of filling the disk
    "train.sigterm_grace_sec": None,       # time a killed train phase gets to save
    "train.emergency_checkpoint": None,    # save state when a run crashes mid-training
    "train.pad_to_multiple_of": None,      # static shapes (TPU default 512)
    "resume.extend": None,                 # keep training after resuming a finished/decayed run
    # ---------------- Time budget ----------------
    "run.mode": None,                      # full | calibration
    "run.calibration_minutes": None,
    "budget.session_hours": None,          # Kaggle: 12 (GPU) / 9 (TPU)
    "budget.safety_margin_min": None,
    "budget.eval_min": None,
    "run.seed": None,
    # ---------------- Data ----------------
    "data.successful_only": None,
    "data.val_fraction": None,
    "data.val_max_examples": None,
    # ---------------- Served agent written into submission.zip ----------------
    "submission.generation.temperature": None,
    "submission.generation.top_p": None,
    "submission.generation.top_k": None,
    "submission.generation.max_output_tokens": None,   # <= 32768
    "submission.generation.thinking_budget": None,     # < max_output_tokens
    "submission.generation.include_thoughts": None,
    "submission.eval_budget.max_time_minutes": None,   # per task; see experiments/exp_006_eval_config_pacing
    "submission.eval_budget.max_tool_calls": None,
    "submission.eval_budget.timeout_seconds": None,
    "submission.eval_budget.max_turns": None,
    # ---------------- Post-train smoke eval (vLLM, CUDA only) ----------------
    "eval.enabled": None,
}

# Per-adapter overrides merged onto the YAML's own overrides, e.g.
# {"tool_lora": {"lora": {"r": 8, "alpha": 16}, "train": {"max_seq_length": 4096}}}
ADAPTER_OVERRIDES = {}


In [ ]:
import json, shutil, subprocess, sys
from pathlib import Path

INPUT = Path("/kaggle/input")
SKIP_DIRS = {"snapshots", "graphs", "embeddings", ".git", "__pycache__"}

def find_dirs(pred, root=INPUT, max_depth=7):
    hits = []
    if not root.exists():
        return hits
    for dirpath, dirnames, filenames in os.walk(root):
        d = Path(dirpath)
        if pred(d, set(filenames)):
            hits.append(d)
            dirnames[:] = []
            continue
        depth = len(d.relative_to(root).parts)
        dirnames[:] = [] if depth >= max_depth else sorted(n for n in dirnames if n not in SKIP_DIRS)
    return hits

def detect_hardware():
    hw = {"tpu": False, "tpu_chips": 0, "gpus": [], "bf16_gpu": None}
    if os.environ.get("TPU_NAME") or os.environ.get("TPU_ACCELERATOR_TYPE") or any(Path("/dev").glob("accel*")):
        hw["tpu"] = True
        try:
            import torch_xla.runtime as xr
            hw["tpu_chips"] = int(xr.global_runtime_device_count())
        except Exception as exc:  # torch_xla missing or runtime not initialised
            hw["tpu_error"] = str(exc)
    try:
        import torch
        if torch.cuda.is_available():
            hw["gpus"] = [
                {"name": torch.cuda.get_device_properties(i).name,
                 "gib": round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 1)}
                for i in range(torch.cuda.device_count())
            ]
            hw["bf16_gpu"] = bool(torch.cuda.is_bf16_supported())
    except ImportError:
        pass
    return hw

PRESETS = {
    "smoke_cpu": {
        "config": "pipeline/multi_lora_sft/configs/e4b_calibration.yaml", "train_hint": None, "served_hint": None, "tiny": True,
        "params": {"run.mode": "calibration", "run.calibration_minutes": 20, "model.quantization": "fp32",
                   "train.accelerator": "cpu", "train.parallelism": "single", "train.grad_accum": 1,
                   "train.max_steps": 2, "train.max_seq_length": 8192, "eval.enabled": False,
                   "parity.num_prompts": 1, "parity.gen_tokens": 4},
    },
    "e4b_gpu_calibration": {"config": "pipeline/multi_lora_sft/configs/e4b_calibration.yaml",
                            "train_hint": "e4b-it-qat-q4_0-unquantized", "served_hint": "e4b-it-qat-w4a16-ct", "params": {}},
    "e4b_gpu_full": {"config": "pipeline/multi_lora_sft/configs/e4b_full.yaml",
                     "train_hint": "e4b-it-qat-q4_0-unquantized", "served_hint": "e4b-it-qat-w4a16-ct", "params": {}},
    "e4b_tpu": {"config": "pipeline/multi_lora_sft/configs/e4b_tpu.yaml",
                "train_hint": "e4b-it-qat-q4_0-unquantized", "served_hint": "e4b-it-qat-w4a16-ct", "params": {}},
    "31b_gpu_full": {"config": "pipeline/multi_lora_sft/configs/31b_full.yaml",
                     "train_hint": "31b-it-qat-q4_0-unquantized", "served_hint": "31b-it-qat-w4a16-ct", "params": {}},
}

HW = detect_hardware()
big_gpus = [g for g in HW["gpus"] if g["gib"] >= 20]
if RUN["preset"] == "auto":
    RUN["preset"] = ("e4b_tpu" if HW["tpu"] else "31b_gpu_full" if len(big_gpus) >= 4
                     else "e4b_gpu_full" if HW["gpus"] else "smoke_cpu")
if RUN["preset"] not in PRESETS:
    raise ValueError(f"Unknown preset {RUN['preset']!r}; choose one of {sorted(PRESETS)}")
PRESET = PRESETS[RUN["preset"]]

COMP = (find_dirs(lambda d, f: {"tasks.jsonl", "HARNESS_README.md"} <= f) or [None])[0]
REPO_SRC = (find_dirs(lambda d, f: (d / "pipeline" / "multi_lora_sft" / "run.py").is_file()) or [Path.cwd()])[0]
MODELS = find_dirs(lambda d, f: "config.json" in f and any(n.endswith(".safetensors") for n in f))
WHEELHOUSES = [w for w in find_dirs(lambda d, f: any(n.endswith(".whl") for n in f)) if not (COMP and w.is_relative_to(COMP))]

def pick_model(hint):
    if not hint:
        return None
    hits = [m for m in MODELS if hint.lower() in str(m).lower()]
    return sorted(hits, key=lambda p: str(p))[-1] if hits else None  # newest version dir sorts last

TRAIN_MODEL = pick_model(PRESET.get("train_hint"))
SERVED_MODEL = pick_model(PRESET.get("served_hint"))

TRAIN_FILES = list(RUN["train_files"])
if not TRAIN_FILES:
    cands = [p for p in INPUT.rglob("*.jsonl") if p.name != "tasks.jsonl" and not (COMP and p.is_relative_to(COMP))] if INPUT.exists() else []
    traj = [p for p in cands if "traj" in p.name.lower() or "sft" in p.name.lower()]
    TRAIN_FILES = [str(p) for p in (traj or cands)]
if not TRAIN_FILES and PRESET.get("tiny") and (REPO_SRC / "data" / "train_trajectories.jsonl").is_file():
    TRAIN_FILES = [str(REPO_SRC / "data" / "train_trajectories.jsonl")]

print(json.dumps({"preset": RUN["preset"], "hardware": HW, "comp": str(COMP), "repo": str(REPO_SRC),
                  "train_model": str(TRAIN_MODEL), "served_model": str(SERVED_MODEL),
                  "train_files": TRAIN_FILES, "wheelhouses": [str(w) for w in WHEELHOUSES]}, indent=2))
if not TRAIN_FILES:
    raise FileNotFoundError("No training JSONL found: attach the trajectory dataset or set RUN['train_files'].")
if not PRESET.get("tiny") and TRAIN_MODEL is None:
    raise FileNotFoundError(f"No base model matching {PRESET['train_hint']!r} under /kaggle/input; attach it as a Kaggle model.")


In [ ]:
import importlib.metadata

OFFICIAL_WH = [w for w in WHEELHOUSES if "developer-agent" in str(w).lower() or "vllm" in " ".join(p.name for p in w.glob("*.whl")).lower()]
PRIVATE_WH = [w for w in WHEELHOUSES if w not in OFFICIAL_WH]
STAGING = Path("/tmp/wheelhouse_staging")
STAGING.mkdir(parents=True, exist_ok=True)

def stage_and_install(wh: Path, no_deps: bool = False):
    dst_dir = STAGING / wh.name
    dst_dir.mkdir(parents=True, exist_ok=True)
    staged = []
    for w in sorted(wh.glob("*.whl")):
        # Restore +cu128 local version suffix if stripped by Kaggle dataset upload
        name = w.name.replace("cu128", "+cu128", 1) if (w.name.startswith("torch-") and "cu128" in w.name and "+cu128" not in w.name) else w.name
        target = dst_dir / name
        if not target.exists():
            shutil.copy2(w, target)
        staged.append(str(target))
    if not staged:
        return
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", str(dst_dir)]
    if no_deps:
        cmd.append("--no-deps")
    cmd.extend(staged)
    subprocess.run(cmd, check=True)

for wh in OFFICIAL_WH:
    print(f"Installing official wheelhouse: {wh}")
    stage_and_install(wh, no_deps=False)
for wh in PRIVATE_WH:
    print(f"Installing training wheelhouse (--no-deps): {wh}")
    stage_and_install(wh, no_deps=True)

for pkg in ("torch", "torch_xla", "transformers", "peft", "trl", "accelerate", "datasets", "bitsandbytes", "vllm", "swegemma", "adk-submission"):
    try:
        print(f"  {pkg:<18} {importlib.metadata.version(pkg)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {pkg:<18} (not installed)")


## Situational adjustments + validation

Builds the `--set` list (preset → PARAMS → RUN["params"] → auto-adjust rules → RUN["set"]), loads it with the real
config validator, and prints the effective per-adapter settings. Nothing trains if this cell raises.


In [ ]:
import math

REPO_WORK = Path("/tmp/GemmaFineTuning")
if REPO_SRC.resolve() != REPO_WORK.resolve():
    shutil.rmtree(REPO_WORK, ignore_errors=True)
    shutil.copytree(REPO_SRC, REPO_WORK, ignore=shutil.ignore_patterns("__pycache__", "*.pyc", ".git", ".venv", "runs", "DataMining"))
os.chdir(REPO_WORK)
if str(REPO_WORK) not in sys.path:
    sys.path.insert(0, str(REPO_WORK))
from pipeline.multi_lora_sft.config import ConfigError, load_config  # noqa: E402

CONFIG_PATH = REPO_WORK / PRESET["config"]
HELPER_KEYS = {"target_effective_batch"}

def as_set(key, val):
    return f"{key}={json.dumps(val)}"

def collect(*layers):
    merged = {}
    for layer in layers:
        merged.update({k: v for k, v in (layer or {}).items() if v is not None})
    return merged

values = collect(PRESET["params"], PARAMS, RUN["params"])
if PRESET.get("tiny"):
    values.update({"model.train_base_path": "__tiny__", "model.served_path": "__tiny__"})
else:
    values["model.train_base_path"] = str(TRAIN_MODEL)
    if SERVED_MODEL is not None:
        values["model.served_path"] = str(SERVED_MODEL)
values["data.train_files"] = TRAIN_FILES
if COMP is not None:
    values["data.tasks_file"] = str(COMP / "tasks.jsonl")
    for sub_key, folder in (("snapshots_dir", "snapshots"), ("graphs_dir", "graphs"), ("embeddings_dir", "embeddings"), ("wheels_dir", "wheels")):
        if (COMP / folder).is_dir():
            values[f"eval.{sub_key}"] = str(COMP / folder)
if RUN.get("eval_task_ids"):
    values["eval.task_ids"] = RUN["eval_task_ids"]
if RUN.get("resume_from"):
    values["run.resume_from"] = str(RUN["resume_from"])
if RUN.get("git_sha"):
    values["run.git_sha"] = RUN["git_sha"]

def to_sets(vals):
    return [as_set(k, v) for k, v in vals.items() if k not in HELPER_KEYS]

cfg0 = load_config(CONFIG_PATH, to_sets(values))
ADJUSTMENTS = []

def adjust(key, val, why):
    values[key] = val
    ADJUSTMENTS.append(f"{key} = {val!r}  <- {why}")

if RUN["auto_adjust"]:
    acc = cfg0.train.accelerator
    gpu_preset = acc == "cuda" or (acc == "auto" and not HW["tpu"] and not PRESET.get("tiny"))
    if HW["tpu"] and cfg0.train.parallelism in {"ddp", "model_split"}:
        raise ConfigError(f"This is a TPU machine but preset {RUN['preset']!r} is CUDA-only; use preset 'e4b_tpu'.")
    if gpu_preset and not HW["gpus"] and not HW["tpu"]:
        raise ConfigError(f"Preset {RUN['preset']!r} needs GPUs but none are visible; use 'smoke_cpu' or attach GPUs.")
    if cfg0.train.parallelism == "model_split" and "31b" in cfg0.effective_served_model_id() and len(big_gpus) < 4:
        raise ConfigError(f"31B QLoRA needs 4 GPUs with >= 20 GiB (e.g. 4xL4); found {HW['gpus']}.")
    if HW["gpus"] and HW["bf16_gpu"] is False:
        print("WARNING: this GPU has no native bf16 (e.g. T4/P100); bf16/NF4 training will be slow or fail. Prefer L4/A100 or TPU.")
    small = [g for g in HW["gpus"] if g["gib"] < 20]
    if small and cfg0.train.max_seq_length > 8192:
        adjust("train.max_seq_length", 8192, f"GPU memory {small[0]['gib']} GiB < 20 GiB")
    if HW["tpu"] and cfg0.train.parallelism == "spmd" and HW["tpu_chips"]:
        chips = HW["tpu_chips"]
        if cfg0.train.per_device_batch % chips:
            adjust("train.per_device_batch", int(math.ceil(cfg0.train.per_device_batch / chips) * chips),
                   f"SPMD batch must be a multiple of {chips} chips")
    target = values.get("target_effective_batch")
    if target:
        pdb = int(values.get("train.per_device_batch", cfg0.train.per_device_batch))
        dp = len(HW["gpus"]) if cfg0.train.parallelism == "ddp" and HW["gpus"] else 1
        adjust("train.grad_accum", max(1, math.ceil(target / (pdb * dp))), f"target_effective_batch={target} (batch {pdb} x dp {dp})")
    limit_h = 9.0 if HW["tpu"] else 12.0
    if cfg0.run.mode == "full" and cfg0.budget.session_hours > limit_h:
        adjust("budget.session_hours", limit_h, f"Kaggle {'TPU' if HW['tpu'] else 'GPU'} session limit")
    out_root = Path(cfg0.run.output_root)
    probe = out_root if out_root.exists() else next((p for p in out_root.parents if p.exists()), Path("/"))
    free_gib = shutil.disk_usage(probe).free / 1024**3
    if free_gib < 10 and cfg0.train.keep_checkpoints > 1:
        adjust("train.keep_checkpoints", 1, f"only {free_gib:.1f} GiB free on {probe}")
    if (HW["tpu"] or not HW["gpus"]) and cfg0.eval.enabled:
        adjust("eval.enabled", False, "smoke eval needs vLLM on CUDA GPUs")

SETS = to_sets(values)
if ADAPTER_OVERRIDES:
    names = [a.name for a in cfg0.adapters]
    for name, ov in ADAPTER_OVERRIDES.items():
        if name not in names:
            raise ConfigError(f"ADAPTER_OVERRIDES: unknown adapter {name!r}; known {names}")
        idx = names.index(name)
        merged = json.loads(json.dumps(cfg0.adapters[idx].overrides or {}))
        for section, sub in ov.items():
            merged.setdefault(section, {}).update(sub)
        SETS.append(as_set(f"adapters.{idx}.overrides", merged))
SETS += [str(s) for s in RUN.get("set", [])]

CFG = load_config(CONFIG_PATH, SETS)  # raises ConfigError with the exact bad key/value
print("Adjustments:\n  " + ("\n  ".join(ADJUSTMENTS) if ADJUSTMENTS else "(none)"))
print(f"\nconfig: {CONFIG_PATH.relative_to(REPO_WORK)}  served: {CFG.effective_served_model_id()}  mode: {CFG.run.mode}  "
      f"session_hours: {CFG.budget.session_hours}  eval: {CFG.eval.enabled}")
print(f"generation: {CFG.submission.generation}\neval_budget: {CFG.submission.eval_budget}\n")
print(f"{'adapter':<12}{'accel':<7}{'parallel':<12}{'quant':<6}{'r':>4}{'alpha':>6}{'lr':>9}{'seq':>7}{'bs':>4}{'ga':>4}{'epochs':>7}{'loss':>10}")
for spec in CFG.adapters:
    e = CFG.resolve_adapter(spec)
    print(f"{e.name:<12}{e.train.accelerator:<7}{e.train.parallelism:<12}{e.model_quantization:<6}{e.lora.r:>4}{e.lora.alpha:>6}"
          f"{e.train.lr:>9.1e}{e.train.max_seq_length:>7}{e.train.per_device_batch:>4}{e.train.grad_accum:>4}{e.train.max_epochs:>7}{e.loss.impl:>10}")


In [ ]:
if RUN["dry_run"]:
    print("dry_run=True: config validated, not launching.")
else:
    cmd = [sys.executable, "-m", "pipeline.multi_lora_sft.run", "--config", str(CONFIG_PATH)]
    for expr in SETS:
        cmd += ["--set", expr]
    print("$ " + " ".join(cmd))
    ORCH_PROC = subprocess.run(cmd)
    print(f"Orchestrator exited with code {ORCH_PROC.returncode}")


In [ ]:
RUNS_ROOT = Path(CFG.run.output_root)
reports = sorted(RUNS_ROOT.glob("*/REPORT.md"), key=lambda p: p.stat().st_mtime) if RUNS_ROOT.exists() else []
if reports:
    latest_report = reports[-1]
    print(f"=== {latest_report} ===\n")
    print(latest_report.read_text(encoding="utf-8"))
    try:
        import matplotlib.pyplot as plt
        for mfile in sorted((latest_report.parent / "metrics").glob("*.jsonl")):
            rows = [json.loads(l) for l in mfile.read_text(encoding="utf-8").splitlines() if l.strip()]
            if rows:
                plt.plot([r["step"] for r in rows], [r["train_loss"] for r in rows], label=mfile.stem)
        plt.xlabel("optimizer step"); plt.ylabel("train loss"); plt.legend(); plt.show()
    except Exception as exc:
        print(f"(loss plot skipped: {exc})")
else:
    print(f"No REPORT.md found under {RUNS_ROOT}")

work_zip = Path(CFG.submission.working_zip_path or "/kaggle/working/submission.zip")
print(f"\nsubmission.zip present: {work_zip.is_file()}" + (f" ({work_zip.stat().st_size / (1024 * 1024):.2f} MiB)" if work_zip.is_file() else ""))
if not RUN["dry_run"] and ORCH_PROC.returncode != 0:
    raise RuntimeError(f"Multi-LoRA SFT pipeline exited with code {ORCH_PROC.returncode}")
